# 05 — Qdrant Store

Tests `infrastructure/db/qdrant_store.py`:
- Singleton client construction
- `collection_info()` — collection exists and has the expected shape
- Ensure points are already ingested (sanity check on count)

> **Requires**: `QDRANT_URL` and `QDRANT_API_KEY` in `.env` and the catalog already ingested.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

assert os.getenv('QDRANT_URL'), 'QDRANT_URL not set'
assert os.getenv('QDRANT_API_KEY'), 'QDRANT_API_KEY not set'
print('Qdrant env vars: OK')

In [ ]:
from infrastructure.db.qdrant_store import get_client, collection_info, ensure_collection
from utils.config import QDRANT_COLLECTION_NAME, QDRANT_EMBEDDING_DIM

## 1. Singleton client

In [ ]:
client1 = get_client()
client2 = get_client()
assert client1 is client2, 'Qdrant client should be a singleton'
print('Singleton: PASSED')

## 2. Collection exists

In [ ]:
client = get_client()
collections = client.get_collections().collections
names = [c.name for c in collections]
print('Collections in Qdrant:', names)

assert QDRANT_COLLECTION_NAME in names, \
    f'Collection "{QDRANT_COLLECTION_NAME}" not found. Run the ingest pipeline first.'
print(f'Collection "{QDRANT_COLLECTION_NAME}": FOUND')

## 3. Collection info shape

In [ ]:
info = collection_info()
print('Collection info:')
for k, v in info.items():
    print(f'  {k}: {v}')

assert 'points_count' in info, 'info must include points_count'
assert 'vector_size' in info, 'info must include vector_size'
assert 'distance' in info, 'info must include distance'
assert 'status' in info, 'info must include status'

assert info['vector_size'] == QDRANT_EMBEDDING_DIM, \
    f"Expected dim {QDRANT_EMBEDDING_DIM}, got {info['vector_size']}"
assert info['distance'].lower() in ('cosine', 'cos'), \
    f"Expected COSINE distance, got {info['distance']}"
print('Collection info shape: PASSED')

## 4. Points are ingested

In [ ]:
info = collection_info()
points_count = info['points_count']
print(f'Points in Qdrant: {points_count:,}')

assert points_count > 1000, \
    f'Expected at least 1000 products ingested, got {points_count}. Run the ingest pipeline.'
print('Points count sanity: PASSED')

## 5. Manual point lookup

In [ ]:
# Retrieve a specific point by ID=0 to verify payload structure
results = client.retrieve(
    collection_name=QDRANT_COLLECTION_NAME,
    ids=[0],
    with_payload=True,
    with_vectors=False,
)

if results:
    payload = results[0].payload
    print('Point 0 payload keys:', list(payload.keys()))
    for k, v in payload.items():
        print(f'  {k}: {str(v)[:80]}')
    
    expected_keys = {'name', 'price', 'category', 'product_url'}
    assert expected_keys.issubset(payload.keys()), \
        f'Missing expected keys. Got: {payload.keys()}'
    print('Payload structure: PASSED')
else:
    print('Point 0 not found — IDs may not start at 0. Skipping structure check.')

## 6. ensure_collection is idempotent

In [ ]:
# Calling ensure_collection when collection already exists should not raise
try:
    ensure_collection()
    print('ensure_collection (idempotent): PASSED')
except Exception as e:
    print(f'ensure_collection raised unexpectedly: {e}')
    raise